In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp

from time_series.data_generators import LorenzGenerator
from time_series.data_handlers import TimeSeriesData
from time_series.models import KernelRidgeRegression, RascuttiModel

import optuna
from time_series.evaluators import MeanSquaredError

from collections import defaultdict
from tqdm import tqdm

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import numpy as np
import shutil


class ExperimentSaver:
    def __init__(self, experiment_name, base_dir="experiments"):
        timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

        self.root = (
            Path(base_dir)
            / experiment_name
            / timestamp
        )

        # core folders
        self.data_dir = self.root / "data"
        self.results_dir = self.root / "results"
        self.models_dir = self.root / "models"
        self.config_dir = self.root / "configs"

        for d in [
            self.root,
            self.data_dir,
            self.results_dir,
            self.models_dir,
            self.config_dir,
        ]:
            d.mkdir(parents=True, exist_ok=True)

    # ---------- saving helpers ----------

    def save_numpy(self, array, name):
        path = self.results_dir / f"{name}.npy"
        np.save(path, array)
        return path

    def save_scores(self, scores: dict, name="scores"):
        path = self.results_dir / f"{name}.json"
        with open(path, "w") as f:
            json.dump(scores, f, indent=2)
        return path

    def save_config(self, config: dict, name="config"):
        path = self.config_dir / f"{name}.json"
        with open(path, "w") as f:
            json.dump(config, f, indent=2)
        return path

    def save_data(self, data, name):
        """
        Saves arbitrary data:
        - numpy arrays -> .npy
        - dicts -> .json
        """
        if isinstance(data, np.ndarray):
            path = self.data_dir / f"{name}.npy"
            np.save(path, data)
        elif isinstance(data, dict):
            path = self.data_dir / f"{name}.json"
            with open(path, "w") as f:
                json.dump(data, f, indent=2)
        else:
            raise TypeError("Unsupported data type")
        return path

    def save_model_file(self, path_to_model, name=None):
        path_to_model = Path(path_to_model)
        if name is None:
            name = path_to_model.name
        dest = self.models_dir / name
        shutil.copy(path_to_model, dest)
        return dest


In [ ]:
class ConfigLog:
    def __init__(self, config=None):
        if not config:
            config = dict()

        self.config = config

    def add_value(self, **kwargs):
        for key, val in kwargs.items():
            self.config[key]=val

    def add_dict(self, dct):
        self.config.update(dct)

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

# MSE vs noise

## Kernel Ridge Regression

In [ ]:
experiment_log = ExperimentSaver("MSE vs Noise - KRR", "experiments")

config_log = ConfigLog()

In [ ]:
n_per = 10
n_groups = 2

np.random.seed(0)

config_log.add_value(n_per=n_per, n_groups=n_groups)

In [ ]:
noise_range = tqdm(np.linspace(1e-3, 10, 20))

for noise in noise_range:
    noise_range.set_description(f"Generating data for noise = {noise}")
    # Generate data groups
    datagroups = []
    for i in range(n_groups):
        # Set group parameters
        rho = np.random.random()*30*2
        sigma = np.random.random()*10*2
        beta = np.random.random()*2*2

        for n in range(n_per):
            x0 = np.random.random(size=3)*10

            generator = LorenzGenerator(
                noise_covariance=noise,
                x0=x0,
                dt=0.01,
                T=10,
                rho=rho,
                sigma=sigma,
                beta=beta
            )

            t, data = generator()

            datagroups.append(
                dict(
                    group = i,
                    sample_id = n,
                    t = t,
                    data = data,
                    data_gen_info = dict(x0=x0, rho=rho, sigma=sigma, beta=beta, noise=noise)
                )
            )

    inner_product_matrix = np.zeros(shape=(len(datagroups), len(datagroups)))
    noise_range.set_description(f"Calculating inner products for noise = {noise}")

    for i in range(len(datagroups)):
        for j in range(i, len(datagroups)):
            data_1 = datagroups[i]["data"]
            data_2 = datagroups[j]["data"]

            dataset_1 = TimeSeriesData(X=data_1[:-1], y=data_1[1:], train_val_test_split=[0.5, 0.2, 0.3], lag=1)
            dataset_2 = TimeSeriesData(X=data_2[:-1], y=data_2[1:], train_val_test_split=[0.5, 0.2, 0.3], lag=1)

            def objective(trial):
                bandwidth = trial.suggest_float("bandwidth", 1e-9, 4)
                reg_1 = trial.suggest_float("reg_1", 1e-12, 1e-2)
                reg_2 = trial.suggest_float("reg_2", 1e-12, 1e-2)

                model1 = KernelRidgeRegression(kernel="rbf", bandwidth=bandwidth, reg=reg_1)
                model2 = KernelRidgeRegression(kernel="rbf", bandwidth=bandwidth, reg=reg_2)

                X_train_1, y_train_1 = dataset_1.train_data()
                X_train_2, y_train_2 = dataset_2.train_data()

                X_test_1, y_test_1 = dataset_1.test_data()
                X_test_2, y_test_2 = dataset_2.test_data()

                model1.fit(X_train_1, y_train_1)
                model2.fit(X_train_2, y_train_2)

                y_pred_1 = model1.predict(X_test_1)
                y_pred_2 = model2.predict(X_test_2)

                return MeanSquaredError()(y_pred_1, y_test_1) + MeanSquaredError()(y_pred_2, y_test_2)
            
            study = optuna.create_study()
            study.optimize(objective, n_trials=5, n_jobs=-1)

            best_params = study.best_params
            bandwidth = study.best_params["bandwidth"]
            reg_1 = study.best_params["reg_1"]
            reg_2 = study.best_params["reg_2"]

            model1 = KernelRidgeRegression(kernel="rbf", bandwidth=bandwidth, reg=reg_1)
            model2 = KernelRidgeRegression(kernel="rbf", bandwidth=bandwidth, reg=reg_2)  

            X_1, y_1 = dataset_1.full_data()
            X_2, y_2 = dataset_2.full_data()

            model1.fit(X_1, y_1)
            model2.fit(X_2, y_2)

            kernels = model1.kernels
            if type(kernels) == list:
                kernel_matrix = sp.linalg.block_diag(
                    *[
                        kernel(X_1, X_2) for kernel in kernels
                    ]
                )  
            else:
                kernel_matrix = sp.linalg.block_diag(
                    *[
                        kernels(X_1, X_2) for i in range(X_1.shape[-1]) # One kernel for each dimension
                    ]
                ) 

            inner_product_matrix[i][j] = model1.alpha.T @ kernel_matrix @ model2.alpha
            inner_product_matrix[j][i] = inner_product_matrix[i][j]

    


    break   